# Behavioural data cleaning and feature engineering

This executed lab applies the Chapter 23–24 sequence: generate relational histories, run auditable cleaning, engineer point-in-time features, and inspect whether the synthetic outcome behaves rationally. The generator is original teaching code, not observed customer data.

In [1]:
from creditriskbook.data import make_behavioral_credit_history
from creditriskbook.data.cleaning import clean_monthly_performance
from creditriskbook.features import build_behavioral_features

case = make_behavioral_credit_history(n_customers=200, months=18, seed=2401)
refs = case.applications[["customer_id", "reference_date"]]
cleaning = clean_monthly_performance(case.monthly_performance, refs)
features = build_behavioral_features(
    cleaning.clean, case.contracts, refs, enquiries=case.bureau_enquiries
)
model_table = case.applications.merge(
    features, on=["customer_id", "reference_date"], validate="one_to_one"
)
print("model table:", model_table.shape)
print("cleaning issues:", len(cleaning.issues))
assert cleaning.issues.empty


model table: (200, 54)
cleaning issues: 0


In [2]:
selected = [
    "max_dpd_6m", "last_dpd", "count_dpd30_6m",
    "count_contracts_last_6m", "current_utilisation",
]
print(model_table[selected].head().round(4).to_string(index=False))


 max_dpd_6m  last_dpd  count_dpd30_6m  count_contracts_last_6m  current_utilisation
       30.0      15.0               1                        2               0.5093
       15.0       0.0               0                        0               0.3330
       15.0       0.0               0                        0               0.8317
       60.0      60.0               2                        1               0.9443
       90.0      90.0               4                        0               1.0407


In [3]:
import pandas as pd

bands = pd.qcut(model_table["max_dpd_6m"], q=4, duplicates="drop")
characteristic = (
    model_table.assign(band=bands)
    .groupby("band", observed=True)["default_12m"]
    .agg(observations="size", defaults="sum", default_rate="mean")
)
print(characteristic.round(4))
assert characteristic["default_rate"].is_monotonic_increasing


(-0.001, 15.0]  n=108  defaults=3  rate=0.0278
(15.0, 60.0]    n=60   defaults=16 rate=0.2667
(60.0, 120.0]   n=28   defaults=13 rate=0.4643


The increasing rate across `max_dpd_6m` bands is a generator rationality check, not evidence that real data must be forced to monotonicity. Real portfolios require source, cohort, policy, uncertainty and stability analysis.